<a href="https://colab.research.google.com/github/yaelezra/ReportAgent/blob/main/ReportAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install smolagents huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 5.5 MB/s eta 0:00:00


In [13]:
!pip install sentence-transformers

In [2]:
import scipy.io as sio
import pandas as pd
from google.colab import userdata
from smolagents import tool, CodeAgent, InferenceClientModel
import numpy as np
from google.colab import output

In [3]:
@tool
def load_mat_file(file_path: str) -> np.array:
    """
    Loads a self-documenting MATLAB sequence file.
    Returns np array of the sequence data.

    Args:
      file_path: The string path to your .mat file.
    """
    # 1. Load the file cleanly using our magic arguments
    mat_contents = sio.loadmat(file_path, struct_as_record=False, squeeze_me=True)

    # 2. Extract the main struct and the parameter values
    seq = mat_contents['sequence_data']

    return seq

In [4]:
@tool
def load_ultrasound_sequence(file_path: str, param_name: str) -> np.array:
    """
    Loads a self-documenting MATLAB sequence file.
    Returns the sequence data for a specific parameter (param_name).

    Args:
      file_path: The string path to your .mat file.
      param_name: The name of the parameter you want to extract.
    """
    # 1. Load the file cleanly using our magic arguments
    mat_contents = sio.loadmat(file_path, struct_as_record=False, squeeze_me=True)

    # 2. Extract the main struct and the parameter values
    seq = mat_contents['sequence_data']
    seq_param = getattr(seq, param_name)

    return seq_param

In [5]:
@tool
def load_ultrasound_doc(file_path: str) -> dict:
      """
      Loads a self-documenting MATLAB sequence file.
      Returns the documentation of all parameters.

      Args:
        file_path: The string path to your .mat file (e.g., 'ultrasound_data.mat').
      """
      # 1. Load the file cleanly using our magic arguments
      mat_contents = sio.loadmat(file_path, struct_as_record=False, squeeze_me=True)

      # 2. Extract the main struct
      seq = mat_contents['sequence_data']

      # 3. Parse the documentation into a readable string
      doc_header = f"--- Documentation for {file_path} ---\n"
      doc_entries = {}

      for field in seq.documentation._fieldnames:
          description = getattr(seq.documentation, field)
          doc_entries[field] = description

      return doc_entries

In [14]:
from sentence_transformers import SentenceTransformer, util

@tool
def find_feature_by_description(file_path: str, query: str, top_k: int = 3) -> dict:
    """
    Finds the most semantically similar features to the query using embedding similarity.
    Use this when the user asks for a feature by a loose or natural language name.
    Returns the top matching field names, their descriptions, and similarity scores.

    Args:
      file_path: The string path to your .mat file.
      query: A natural language description of the feature you're looking for.
      top_k: Number of top matches to return (default 3).
    """
    mat_contents = sio.loadmat(file_path, struct_as_record=False, squeeze_me=True)
    seq = mat_contents['sequence_data']

    # Build corpus: "field_name: description"
    fields, corpus = [], []
    for field in seq._fieldnames:
        if field == 'documentation':
            continue
        description = ''
        if field in seq.documentation._fieldnames:
            description = str(getattr(seq.documentation, field))
        fields.append(field)
        corpus.append(f"{field}: {description}")

    model = SentenceTransformer('all-MiniLM-L6-v2')
    query_emb  = model.encode(query, convert_to_tensor=True)
    corpus_embs = model.encode(corpus, convert_to_tensor=True)

    scores = util.cos_sim(query_emb, corpus_embs)[0]
    top = scores.topk(min(top_k, len(fields)))

    results = {}
    for score, idx in zip(top.values, top.indices):
        field = fields[idx]
        results[field] = {
            "description": corpus[idx],
            "similarity":  round(float(score), 3)
        }

    return results

In [23]:
@tool
def find_feature_in_files(file_paths: list, param_name: str) -> list:
  """
  Find the same feature value in different files.

  Args:
    file_paths: TThe list of the mat files.
    param_name: The name of the parameter you want to extract.
  """
  file_features = []
  for path in file_paths:
    seq = load_mat_file(path)
    file_features.append(seq[param_name])
  return file_features



In [24]:
# Assuming your tools and model are already initialized from the previous step...

# Load your secure token
hf_token = userdata.get('HF_TOKEN')

# Initialize the model
model = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-7B-Instruct", token=hf_token)

agent = CodeAgent(
    tools=[load_mat_file, load_ultrasound_sequence, load_ultrasound_doc, find_feature_by_description, find_feature_in_files],
    model=model,
    additional_authorized_imports=["numpy", "pandas", "seaborn", "matplotlib.pyplot", "math"]
)

def run_custom_agent(file_paths: list, instructions: str, user_prompt: str):
    """
    Feeds strict instructions and a user prompt to the agent.
    """
    # We combine the instructions and the prompt into one super-prompt
    full_prompt = f"""
    You are given a list of files - each file contains a struct in which each field is a vector of values in time.
    {file_paths} is the list of file paths.
    SYSTEM INSTRUCTIONS TO FOLLOW STRICTLY:
    {instructions}

    --------------------------------------------------
    USER REQUEST:
    {user_prompt}
    """

    print("🤖 Agent is thinking...\n")
    response = agent.run(full_prompt)

    print("\n🎯 FINAL ANSWER:")
    print(response)
    return response


In [ ]:
# ==========================================
# HOW TO USE IT
# ==========================================

# 1. Define your strict rules (You can change these whenever you want!)
my_rules = """
- You are a senior Data Scientist.
- You know how to analyze data, give insights, and make graphs and reports.
- You must always explain the meaning of a parameter before showing its data.
- Never show the logs and codes, just the final answer and the code that created it.
- In the end, tell which tools you used to answer the question.
- If you encounter an error that you can't resolve, stop and print the error as your response.
- If I ask you to make a report you should make a full analysis about all the parameters and the relationship between them.
- If you can't find a name of a feature that you need, look for it in the documentation using the find_feature_by_description tool.
- If you make plots, graphs or reports, show and save them.
"""

# 2. Define the specific question you want to ask right now
my_question = "Print the feature1 in each file"

# 3. Run the agent with both!
file_path = ['/content/ultrasound_sequence_features_example1.mat', '/content/ultrasound_sequence_features_example1.mat']
response = run_custom_agent(file_path, instructions=my_rules, user_prompt=my_question)

🤖 Agent is thinking...



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are given a list of files - each file contains a struct in which each field is a vector of values in time.  │
│     ['/content/ultrasound_sequence_features_example1.mat',                                                      │
│ '/content/ultrasound_sequence_features_example1.mat'] is the list of file paths.                                │
│     SYSTEM INSTRUCTIONS TO FOLLOW STRICTLY:                                                                     │
│                                                                                                                 │
│ - You are a senior Data Scientist.                                                                              │
│ - You know how to analyze data, give insights, and make graphs and reports.                                     │
│ - You must always explain the meaning of a parameter before showing its data.                                   │
│ - Never show the logs and codes, just the final answer and the code that created it.                            │
│ - In the end, tell which tools you used to answer the question.                                                 │
│ - If you encounter an error that you can't resolve, stop and print the error as your response.                  │
│ - If I ask you to make a report you should make a full analysis about all the parameters and the relationship   │
│ between them.                                                                                                   │
│ - If you can't find a name of a feature that you need, look for it in the documentation using the               │
│ find_feature_by_description tool.                                                                               │
│ - If you make plots, graphs or reports, show and save them.                                                     │
│                                                                                                                 │
│                                                                                                                 │
│     --------------------------------------------------                                                          │
│     USER REQUEST:                                                                                               │
│     Print the feature1 in each file                                                                             │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-7B-Instruct ─────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  file_paths = ['/content/ultrasound_sequence_features_example1.mat']                                              
                                                                                                                   
  # Find the description of 'feature1' in the files                                                                
  features_info = find_feature_by_description(file_paths=file_paths, query="feature1")                             
                                                                                                                   
  # Extract the field name and index                                                                               
  feature_name = features_info[0][0]                                                                               
  feature_index = 0                                                                                                
                                                                                                                   
  # Load and print 'feature1' for every file in the list                                                           
  for file_path in file_paths:                                                                                     
      feature_data = load_mat_file(file_path=[file_path])[feature_name]                                            
      print(f"Data for '{feature_name}' in {file_path}:")                                                          
      print(feature_data)                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution failed at line 'features_info = find_feature_by_description(file_paths=file_paths, 
query="feature1")' due to: TypeError: find_feature_by_description() got an unexpected keyword argument 'file_paths'

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 8 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import numpy as np                                                                                               
                                                                                                                   
  file_paths = ['/content/ultrasound_sequence_features_example1.mat']                                              
                                                                                                                   
  # Retrieve documentation for each file and extract the actual features                                           
  docs = [load_mat_file(file_path=file_path) for file_path in file_paths]                                          
  extracted_features = [list(doc.keys())[0] for doc in docs]                                                       
                                                                                                                   
  # Create a dictionary to hold the feature names                                                                  
  features_dict = {file_path.split('/')[-1]: feature for file_path, feature in zip(file_paths,                     
  extracted_features)}                                                                                             
                                                                                                                   
  # Check the dictionary to ensure all necessary features are present                                              
  if "feature1" not in features_dict:                                                                              
      raise KeyError("Feature1 not found in the loaded files.")                                                    
                                                                                                                   
  # Assuming feature names are consistent across the files                                                         
  actual_feature_name = "feature1"                                                                                 
                                                                                                                   
  # Extract and print 'feature1' values for each file                                                              
  for feature_name, file_path in features_dict.items():                                                            
      data = load_mat_file(file_path=[file_path])                                                                  
      feature_values = data[actual_feature_name]                                                                   
      print(f"Values for '{actual_feature_name}' in {feature_name}: ")                                             
      print(feature_values)                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution failed at line 'extracted_features = [list(doc.keys())[0] for doc in docs]' due to: 
InterpreterError: Object <scipy.io.matlab._mio5_params.mat_struct object at 0x7b11689d5b80> has no attribute keys

[Step 8: Duration 7.01 seconds| Input tokens: 36,830 | Output tokens: 2,194]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 9 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━